In [2]:
#DATA PREPROCESSING
# Reload necessary libraries
import pandas as pd

# Load the anime dataset
anime_file_path ="/content/anime.csv"
anime_df = pd.read_csv(anime_file_path)

# Display basic information about the dataset
anime_df.info(), anime_df.head()


# Handle missing values
anime_df["genre"].fillna("Unknown", inplace=True)
anime_df["type"].fillna("Unknown", inplace=True)
anime_df["rating"].fillna(anime_df["rating"].median(), inplace=True)

# Convert 'episodes' column to numeric (handling 'Unknown' values)
anime_df["episodes"] = pd.to_numeric(anime_df["episodes"], errors='coerce')
anime_df["episodes"].fillna(anime_df["episodes"].median(), inplace=True)  # Fill missing with median

# Confirm changes
anime_df.info(), anime_df.head()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12294 non-null  object 
 3   type      12294 non-null  object 
 4   episodes  12294 non-null  float64
 5   rating    12294 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(2)

<ipython-input-2-c8472d9f473d>:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  anime_df["genre"].fillna("Unknown", inplace=True)
<ipython-input-2-c8472d9f473d>:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usi

(None,
    anime_id                              name  \
 0     32281                    Kimi no Na wa.   
 1      5114  Fullmetal Alchemist: Brotherhood   
 2     28977                          Gintama°   
 3      9253                       Steins;Gate   
 4      9969                     Gintama&#039;   
 
                                                genre   type  episodes  rating  \
 0               Drama, Romance, School, Supernatural  Movie       1.0    9.37   
 1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV      64.0    9.26   
 2  Action, Comedy, Historical, Parody, Samurai, S...     TV      51.0    9.25   
 3                                   Sci-Fi, Thriller     TV      24.0    9.17   
 4  Action, Comedy, Historical, Parody, Samurai, S...     TV      51.0    9.16   
 
    members  
 0   200630  
 1   793665  
 2   114262  
 3   673572  
 4   151266  )

In [3]:
#FEATURE EXTRACTION
from sklearn.preprocessing import MinMaxScaler

# One-hot encode 'genre' by splitting multiple genres
genres_split = anime_df['genre'].str.get_dummies(sep=", ")

# Normalize 'rating', 'episodes', and 'members'
scaler = MinMaxScaler()
anime_df[['rating', 'episodes', 'members']] = scaler.fit_transform(anime_df[['rating', 'episodes', 'members']])

# Combine numerical and genre features for similarity computation
anime_features = pd.concat([anime_df[['rating', 'episodes', 'members']], genres_split], axis=1)

# Display transformed data
anime_features.head()


,rating,episodes,members,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,...,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Unknown,Vampire,Yaoi,Yuri
0,0.924370,0.000000,0.197872,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
1,0.911164,0.034673,0.782770,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,0.909964,0.027518,0.112689,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0.900360,0.012658,0.664325,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,0.899160,0.027518,0.149186,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
# RECOMMENDATION SYSTEM
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix
cosine_sim = cosine_similarity(anime_features)

# Convert similarity matrix to DataFrame for easier lookup
anime_sim_df = pd.DataFrame(cosine_sim, index=anime_df["name"], columns=anime_df["name"])

# Display a small sample of the similarity matrix
anime_sim_df.iloc[:5, :5]


name,Kimi no Na wa.,Fullmetal Alchemist: Brotherhood,Gintama°,Steins;Gate,Gintama&#039;
name,,,,,
Kimi no Na wa.,1.000000,0.310682,0.139386,0.241574,0.139028
Fullmetal Alchemist: Brotherhood,0.310682,1.000000,0.358634,0.255865,0.361165
Gintama°,0.139386,0.358634,1.000000,0.375156,0.999908
Steins;Gate,0.241574,0.255865,0.375156,1.000000,0.378272
Gintama&#039;,0.139028,0.361165,0.999908,0.378272,1.000000


In [14]:
# EVALUATION
 # SPILLTING DATSET
from sklearn.model_selection import train_test_split

# Create a user-anime interaction matrix (simulated)
# Assuming 'anime_id' is a unique identifier for each anime
# and 'rating' represents user ratings (replace if different)
#--- Changed from "anime_id" to "anime_id" for index, as "mal_id" is not a column in the dataframe
# If you intend to use a different unique identifier, replace "anime_id" with the correct column name
user_anime_matrix = anime_df.pivot_table(index="anime_id", columns="name", values="rating")

# Fill NaN values with 0 (assuming unwatched anime)
user_anime_matrix = user_anime_matrix.fillna(0)

# Split the dataset into train and test (hide some interactions)
train, test = train_test_split(user_anime_matrix, test_size=0.2, random_state=42)

# GENERATING RECOMMENDATION

def recommend_anime_for_user(user_id, train_matrix, anime_sim_df, top_n=5):
    """Recommend top N anime for a given user based on item similarity."""
    #--- accessing the row of the train_matrix using user_id in loc
    user_ratings = train_matrix.loc[user_id]  # Get user's anime ratings
    seen_anime = user_ratings[user_ratings > 0].index  # Anime the user has watched

    recommendations = {}
    for anime in seen_anime:
        similar_anime = anime_sim_df[anime].sort_values(ascending=False).iloc[1:top_n+1]
        for rec, score in similar_anime.items():
            if rec not in seen_anime:
                recommendations[rec] = recommendations.get(rec, 0) + score  #

In [25]:
#EVALUATING PERFORMANCE (Precision, Recall, and F1-Score)
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_recommendations(test_matrix, train_matrix, anime_sim_df, top_n=5):
    """Evaluate recommendation performance using Precision, Recall, and F1-score."""
    precision_list, recall_list, f1_list = [], [], []

    for user_id in test_matrix.index:
       # Check if user_id is in train_matrix to avoid KeyError
        if user_id not in train_matrix.index:
            continue  # Skip if user_id is not in train_matrix

        actual_watched = set(test_matrix.loc[user_id][test_matrix.loc[user_id] > 0].index)
        recommended = {anime for anime, _ in recommend_anime_for_user(user_id, train_matrix, anime_sim_df, top_n)}

        if len(actual_watched) == 0 or len(recommended) == 0:
            continue  # Skip users with no test data

        tp = len(actual_watched & recommended)  # True positives (correct recommendations)
        fp = len(recommended - actual_watched)  # False positives (incorrect recommendations)
        fn = len(actual_watched - recommended)  # False negatives (missed relevant anime)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)
        print(f"User ID: {user_id}")
        print(f"Actual Watched: {actual_watched}")
        print(f"Recommended: {recommended}")
    # The return statement was improperly indented
    return {
        "Precision": sum(precision_list) / len(precision_list) if precision_list else 0,  # Return 0 if precision_list is empty
        "Recall": sum(recall_list) / len(recall_list) if recall_list else 0,          # Return 0 if recall_list is empty
        "F1-score": sum(f1_list) / len(f1_list) if f1_list else 0                      # Return 0 if f1_list is empty
    }


# Evaluate the recommendation system
evaluate_recommendations(test_matrix=test, train_matrix=train, anime_sim_df=anime_sim_df)

{'Precision': 0, 'Recall': 0, 'F1-score': 0}

# **Analyzing Results**
The function will return the average precision, recall, and F1-score across all users.

 **Interpretation:**


* High Precision → Recommendations are mostly correct.
* High Recall → Most relevant anime are recommended.
* High F1-score → Good balance between precision and recall.

**Areas for Improvement:**

* Adjust the similarity calculation (e.g., using Pearson correlation instead of cosine similarity).
* Use hybrid filtering (combine content-based + collaborative filtering).
* Experiment with different thresholds for cosine similarity scores.


# **INTERVIEW QUESTION**
**1. Can you explain the difference between user-based and item-based collaborative filtering?**

ANS: User-based collaborative filtering recommends items based on the preferences of users who have similar tastes, while item-based filtering recommends items based on the similarity of items to those a user liked in the past.

 User-based collaborative filtering:
 * Focus:
Predicts a user's preferences for a particular item by examining the ratings given to that item by other users with similar tastes.
 * Process:
Identifies similar users based on their past interactions (ratings, purchases, etc.), and then uses the recommendations of those similar users to suggest items to the target user.
 * Example:
If you and User X have rated the same movies similarly, the system might recommend movies User X liked that you haven't yet seen.

Item-based collaborative filtering:
 * Focus:
Predicts a user's potential interest in an item by considering the similarity between that item and other items the user liked in the past.
 * Process:
Determines which items are similar based on ratings given by users, and suggests similar items to the target user based on items they have already rated highly.
 * Example:
If you liked Movie A and Movie B, the system might suggest movies that are similar to Movie A and Movie B.



  
**2. What is collaborative filtering, and how does it work?**

ANS: Collaborative filtering is a recommendation technique that predicts a user's interests by analyzing preferences from multiple users. It works in two main ways:

* User-Based Collaborative Filtering

 Step 1: Find users with similar preferences.

 Step 2: Recommend items that similar users liked but the target user hasn't seen yet.
* Item-Based Collaborative Filtering

 Step 1: Identify similar items based on user interactions.
 Step 2: Recommend items similar to those the user has already engaged with.

 Real-World Example:

 Netflix uses user-based filtering to suggest shows based on similar viewers' watch history.

 Amazon uses item-based filtering to recommend products similar to those you've viewed.
